<div style="border-left:4px solid #fbbf24;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#fbbf24;">Run</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">The whole pipeline end to end, then the live service.</div></div>

<div style="font:400 15px/1.65 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#3f3f46;">The whole pipeline on one question, every step visible, then the same pipeline behind a public address so it can be used from a browser.</div>

In [ ]:
# The code comes from GitHub. The repository is private, so this needs a
# GITHUB_TOKEN secret (Add-ons -> Secrets).
import os, subprocess, sys
from pathlib import Path

ROOT = Path("/kaggle/working/nl2sql")
if not ROOT.exists():
    from kaggle_secrets import UserSecretsClient
    try:
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        # Kaggle answers 400 here when the label is not attached to this notebook,
        # and the client reports that as a connection error sixty frames deep.
        raise SystemExit(
            f"GITHUB_TOKEN could not be read ({type(e).__name__}: {e}). "
            "Open this notebook on Kaggle and check Add-ons -> Secrets: the secret "
            "must exist and be attached here. Internet must be on as well."
        ) from e
    url = "https://github.com/Kirazul/NL2SQL-demo.git".replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", url, str(ROOT)], check=True)
    # git writes the clone URL into .git/config, token and all, and Kaggle saves
    # .git with the notebook output. Put the plain address back immediately.
    subprocess.run(["git", "-C", str(ROOT), "remote", "set-url", "origin", "https://github.com/Kirazul/NL2SQL-demo.git"], check=True)

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("code:", ROOT)

In [ ]:
# pip writes to site-packages, which is not part of notebook 1's saved output, so
# every session installs again. Nearly all of it is already in the Kaggle image.
!pip install -q -e . 2>&1 | tail -2
print("dependencies ready")

In [ ]:
# Notebook 1 built the database, the index and the model weights and saved them
# with its output. Kaggle mounts that output read-only under /kaggle/input, and
# every one of the three is opened read-only here too - so point the settings at
# the mount rather than copying two gigabytes into the working directory.
#
# How deep inside the mount they sit depends on what notebook 1's working
# directory held when it was saved. Matching one guessed shape reported a good
# output as a missing database, so search the plausible depths instead, and when
# nothing turns up print what is actually mounted rather than assert a cause.
MOUNTS = sorted(p for p in Path("/kaggle/input").glob("*") if p.is_dir())
found = [
    hit
    for m in MOUNTS
    for pattern in ("data/eicu.db", "*/data/eicu.db", "*/*/data/eicu.db")
    for hit in sorted(m.glob(pattern))
]
if not found:
    print("mounted under /kaggle/input:" if MOUNTS else "nothing is mounted under /kaggle/input")
    for m in MOUNTS:
        print(" ", m.name + "/", " ".join(sorted(c.name for c in m.iterdir())[:8]))
    raise SystemExit(
        "No eicu.db under any of them. Notebook 1's saved output is what carries it: "
        "open the sidebar, Input -> Add Input -> Your Work -> NL2SQL 1 Setup, and check "
        "the version it pins is one that ran to the end."
    )
SETUP = found[0].parents[1]

# Name a missing piece here rather than several cells later, from inside whichever
# library opens it first.
for path in (SETUP / "data/index.db", SETUP / "models/gliner2-base-v1"):
    if not path.exists():
        print("missing from notebook 1's output:", path)

# Set before nl2sql is imported anywhere: settings() is read once and cached. A
# subprocess started later - the API server in notebook 5 - inherits these too.
os.environ["DB_PATH"] = str(SETUP / "data" / "eicu.db")
os.environ["INDEX_PATH"] = str(SETUP / "data" / "index.db")
os.environ["GLINER_MODEL"] = str(SETUP / "models" / "gliner2-base-v1")
weights = sorted(SETUP.glob("models/*/*.gguf"))
if weights:
    os.environ["LOCAL_GGUF_PATH"] = str(weights[0])

for name in ("DB_PATH", "INDEX_PATH", "GLINER_MODEL", "LOCAL_GGUF_PATH"):
    print(f"  {name:<16} {os.environ.get(name, 'missing - the steps that need it will say so')}")

In [ ]:
# Only the pages that run a model on this machine need llama-cpp-python, and
# building it costs several minutes. Notebook 1 keeps the wheel it built with its
# output, so installing from that is a copy. Runs after the cell above, so a mount
# that turned out to be unusable stops the notebook before the compile, not after.
wheelhouse = next(
    (w.parent for m in MOUNTS for pattern in ("wheels/*.whl", "*/wheels/*.whl")
     for w in m.glob(pattern)),
    None,
)
if wheelhouse:
    !pip install -q --find-links {wheelhouse} llama-cpp-python 2>&1 | tail -2
else:
    print("no wheel in notebook 1's output - building from source, a few minutes")
    !pip install -q llama-cpp-python 2>&1 | tail -2

try:
    import llama_cpp
    print("local model runtime: llama-cpp-python", llama_cpp.__version__)
except ImportError as e:
    print("llama-cpp-python is unavailable:", e)
    print("Full Local will fail, and the answer writer will show a plain table instead.")

In [ ]:
# Keys live in Kaggle secrets, never in the notebook.
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("GROQ_API_KEY", "OPENROUTER_API_KEY", "LANGSMITH_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print(f"{name} not set - the steps that need it will say so")

os.environ["LANGSMITH_TRACING"] = "1"
os.environ["LANGSMITH_PROJECT"] = "nl2sql"

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#fbbf24;">1.</span> One question, end to end</div></div>

In [ ]:
from nl2sql.core import graph

state = graph.run('How many patients over 65 received aspirin?')
print(state["answer"])

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#fbbf24;">2.</span> Every step it took</div></div>

In [ ]:
for step in state.get("trace", []):
    zone = "cloud" if step["zone"] == "cloud" else "here "
    print(f"  [{zone}] {step['label']:<42} {step['ms']:>7.0f} ms  {step['summary']}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#fbbf24;">3.</span> What left the machine</div></div>

In [ ]:
print("characters sent  :", state.get("egress_chars"))
print("real values sent :", state.get("egress_values"))
print("query            :", state.get("sql"))
print()
from nl2sql.privacy import audit
print(audit.report())

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#fbbf24;">4.</span> The service</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">FastAPI, started in the background.</div></div>

In [ ]:
import subprocess, time

import httpx

# Keep the log: a server that dies on import says why here, and nowhere else.
log = open(ROOT / "uvicorn.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "nl2sql.api:app", "--host", "0.0.0.0", "--port", "7860"],
    cwd=ROOT, stdout=log, stderr=subprocess.STDOUT,
)

# Ask until it answers rather than sleeping a guessed number of seconds: loading
# the local weights is what takes the time, and it varies.
health = None
for _ in range(60):
    if server.poll() is not None:
        raise SystemExit("the server exited:\n" + (ROOT / "uvicorn.log").read_text()[-2000:])
    try:
        health = httpx.get("http://127.0.0.1:7860/health", timeout=5).json()
        break
    except Exception:
        time.sleep(2)

if health is None:
    raise SystemExit("the server did not answer:\n" + (ROOT / "uvicorn.log").read_text()[-2000:])
print(health)

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#fbbf24;">5.</span> A public address</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">A tunnel, because a Kaggle session is not reachable from outside on its own.</div></div>

In [ ]:
!pip install -q pycloudflared 2>&1 | tail -1
from pycloudflared import try_cloudflare

tunnel = try_cloudflare(port=7860)
print("public address:", tunnel.tunnel)

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#fbbf24;">6.</span> Tell the front end where to find it</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">The page at https://nl2sql.eclipse-kira.workers.dev keeps one key: the address the pipeline is currently on.</div></div>

In [ ]:
import httpx

response = httpx.post(
    "https://nl2sql.eclipse-kira.workers.dev/api/backend",
    json={"url": tunnel.tunnel},
    timeout=30,
)
print(response.status_code, response.text)
print("\nopen https://nl2sql.eclipse-kira.workers.dev")

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;border-top:1px solid #e4e4e7;padding-top:12px;margin-top:26px;">Leave this notebook running while the demo is in use. When the session stops, the address stops with it.</div>